# back-fn-call-with-recipe-args — faded example 1: Complete the canonical back_fn call for add_back0

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `back-fn-call-with-recipe-args`. The last cell reports your progress on the `Backprop: back fn call with recipe args` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: back fn call with recipe args` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`back-fn-call-with-recipe-args`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "back-fn-call-with-recipe-args"
DD_SUBTOPIC = "Backprop: back fn call with recipe args"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The canonical invocation is `back_fn(grad_out, node.array, *recipe.args, **recipe.kwargs)`: grad_out first, then the cached out, then the splatted positional args and kwargs. For `add(x, y)`, `add_back0` returns `grad_out` unchanged (derivative of a sum w.r.t. either input is 1).

## Faded exercise 1

### Faded — finish the canonical four-channel back_fn call

A node was produced by `add(x, y)`. We want the gradient into parent `argnum=0`. The back_fn registry and `add_back0` are written for you. Complete the single line that **invokes the looked-up back_fn with the canonical argument shape** — grad_out, the cached `out`, then the splatted recipe args and kwargs. Return its result.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def add(x, y):
    return x + y

def add_back0(grad_out, out, x, y):
    return grad_out.clone()

BACK_FUNCS = {(add, 0): add_back0}

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

def backward_into_parent(node, grad_out, argnum):
    back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
    result = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
    return result

t.manual_seed(0)
x = t.randn(2, 2)
y = t.randn(2, 2)
out = add(x, y)
node = Node(out, Recipe(add, (x, y), {}))

def _test():
    grad_out = t.randn(2, 2)
    dx = backward_into_parent(node, grad_out, 0)
    assert isinstance(dx, t.Tensor), 'must return a torch.Tensor'
    assert tuple(dx.shape) == (2, 2), f'wrong shape {tuple(dx.shape)}'
    # d/dx (x + y) = 1, so grad into x equals grad_out exactly
    assert t.allclose(dx, grad_out), 'add_back0 should pass grad_out through unchanged'


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def add(x, y):
    return x + y

def add_back0(grad_out, out, x, y):
    return grad_out.clone()

BACK_FUNCS = {(add, 0): add_back0}

class Recipe:
    def __init__(self, func, args, kwargs):
        self.func, self.args, self.kwargs = func, args, kwargs

class Node:
    def __init__(self, array, recipe):
        self.array, self.recipe = array, recipe

def backward_into_parent(node, grad_out, argnum):
    back_fn = BACK_FUNCS[(node.recipe.func, argnum)]
    result = back_fn(grad_out, node.array, *node.recipe.args, **node.recipe.kwargs)
    return result

t.manual_seed(0)
x = t.randn(2, 2)
y = t.randn(2, 2)
out = add(x, y)
node = Node(out, Recipe(add, (x, y), {}))
```
</details>